In [1]:
import pandas as pd
from data_utils.data_dir import DataDir
from pathlib import Path
from baseline.aggregated_features_baseline.constants import EventTypes
from data_utils.utils import (
    load_with_properties,
)


In [2]:
data_dir = DataDir(Path("data"))

product_buy_df = load_with_properties(
    data_dir=data_dir, event_type=EventTypes.PRODUCT_BUY.value
)

product_buy_df.head()

,client_id,timestamp,sku,category,price,name
0,17649961,2022-07-23 20:15:25,18485,5492,72,[187 47 120 237 234 172 91 172 67 153 32 ...
1,16696114,2022-07-11 16:31:30,81192,6519,99,[241 241 241 241 241 241 241 241 112 241 241 2...
2,10238779,2022-05-29 19:35:40,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
3,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...
4,10238779,2022-05-29 19:38:05,510014,6350,58,[167 24 193 24 24 92 167 76 29 45 172 2...


In [3]:
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Get top n most popular categories
def get_top_n_categories(df, n):
    return df['category'].value_counts().head(n).index.tolist()

# 2. Get top n most popular items (skus)
def get_top_n_items(df, n):
    return df['sku'].value_counts().head(n).index.tolist()

# 3. Create one-hot encoded interaction DataFrame
def create_user_item_category_ohe(df, top_n_items, top_n_categories, binary=True):
    df_filtered = df.copy()
    
    # Keep only top N items/categories
    df_filtered['item_interacted'] = df_filtered['sku'].apply(lambda x: x if x in top_n_items else None)
    df_filtered['cat_interacted'] = df_filtered['category'].apply(lambda x: x if x in top_n_categories else None)
    
    # One-hot encode
    item_ohe = pd.get_dummies(df_filtered['item_interacted'], prefix='item')
    cat_ohe = pd.get_dummies(df_filtered['cat_interacted'], prefix='category')
    
    # Combine with user IDs
    user_ohe = pd.concat([df_filtered[['client_id']], item_ohe, cat_ohe], axis=1)
    
    # Group and either sum (counts) or max (binary)
    agg_func = 'max' if binary else 'sum'
    user_ohe = user_ohe.groupby('client_id').agg(agg_func).reset_index()
    
    return user_ohe

In [4]:
top_n = 100  # or any number you want
df = product_buy_df
top_categories = get_top_n_categories(df, top_n)
top_items = get_top_n_items(df, top_n)

user_ohe_df = create_user_item_category_ohe(df, top_items, top_categories, binary=False)


In [5]:
import numpy as np
def convert_name_to_array(name_str):
    # Remove brackets and split into numbers
    return np.fromstring(name_str.strip('[]'), sep=' ')

def average_array_per_user(df, target_col="name"):
    # Convert string to numpy array
    df['array'] = df[target_col].apply(convert_name_to_array)
    
    # Group by client_id and average arrays
    avg_name_vectors = (
        df.groupby('client_id')['array']
        .apply(lambda arrays: np.mean(list(arrays), axis=0))
    )
    
    # Convert Series of arrays to DataFrame
    name_df = pd.DataFrame(avg_name_vectors.tolist(), index=avg_name_vectors.index)
    name_df.columns = [f'{target_col}_feat_{i}' for i in range(name_df.shape[1])]
    name_df.reset_index(inplace=True)

    return name_df

name_df = average_array_per_user(df)

In [6]:
name_df

,client_id,name_feat_0,name_feat_1,name_feat_2,name_feat_3,name_feat_4,name_feat_5,name_feat_6,name_feat_7,name_feat_8,name_feat_9,name_feat_10,name_feat_11,name_feat_12,name_feat_13,name_feat_14,name_feat_15
0,14,152.500000,167.0,207.500000,195.000000,195.000000,122.000000,172.0,167.000000,83.500000,116.000000,33.000000,21.000000,221.000000,192.000000,169.0,220.000000
1,28,118.000000,22.0,118.000000,118.000000,118.000000,118.000000,22.0,22.000000,118.000000,202.000000,118.000000,118.000000,22.000000,118.000000,118.0,118.000000
2,54,138.333333,116.0,140.111111,114.333333,105.777778,163.111111,170.0,59.555556,60.444444,186.111111,63.777778,144.888889,103.666667,128.666667,116.0,124.888889
3,70,79.000000,228.0,67.000000,86.000000,131.000000,76.000000,38.0,79.000000,121.000000,12.000000,210.000000,45.000000,41.000000,159.000000,45.0,121.000000
4,90,19.000000,157.0,71.000000,157.000000,232.000000,157.000000,167.0,47.000000,157.000000,157.000000,157.000000,157.000000,127.000000,157.000000,157.0,244.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
618887,23875113,38.000000,250.0,67.000000,168.000000,191.000000,160.000000,160.0,143.000000,145.000000,131.000000,47.000000,148.000000,148.000000,237.000000,38.0,28.000000
618888,23875168,222.000000,59.0,52.000000,128.000000,175.000000,214.000000,87.0,156.000000,156.000000,67.000000,124.000000,71.000000,130.000000,108.000000,125.0,108.000000
618889,23875176,64.000000,20.0,23.000000,222.000000,117.000000,114.000000,171.0,64.000000,97.000000,119.000000,81.000000,98.000000,29.000000,50.000000,180.0,66.000000
618890,23875208,217.000000,234.5,154.500000,193.500000,122.000000,150.000000,174.5,199.000000,48.500000,171.500000,174.500000,166.000000,73.500000,105.500000,87.0,181.500000


In [7]:
search_query = load_with_properties(
    data_dir=data_dir, event_type=EventTypes.SEARCH_QUERY.value
)
query_df = average_array_per_user(search_query, target_col="query")

In [8]:
merged = pd.merge(query_df, name_df, how='outer')
merged = pd.merge(merged, user_ohe_df, how="outer")


In [9]:
df_filled = merged.fillna(0)
df_filled.head()

,client_id,query_feat_0,query_feat_1,query_feat_2,query_feat_3,query_feat_4,query_feat_5,query_feat_6,query_feat_7,query_feat_8,...,category_5979.0,category_6214.0,category_6393.0,category_6440.0,category_6508.0,category_6771.0,category_6821.0,category_6839.0,category_6857.0,category_6899.0
0,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,61,170.0,152.0,3.0,222.0,170.0,121.0,121.0,84.0,121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
client_ids = df_filled["client_id"].values

In [11]:
client_embeddings = df_filled.drop(columns=["client_id"]).values.astype("float16")
client_embeddings

array([[  0. ,   0. ,   0. , ...,   0. ,   0. ,   0. ],
       [  0. ,   0. ,   0. , ...,   0. ,   0. ,   0. ],
       [  0. ,   0. ,   0. , ...,   0. ,   0. ,   0. ],
       ...,
       [104.2, 185.2, 151.6, ...,   0. ,   0. ,   0. ],
       [  0. ,   0. ,   0. , ...,   0. ,   0. ,   0. ],
       [  0. ,   0. ,   0. , ...,   0. ,   0. ,   0. ]],
      shape=(1420592, 232), dtype=float16)

In [12]:
from baseline.aggregated_features_baseline.create_embeddings import (
    save_embeddings,
)

save_embeddings(Path("embeddings_handmade"), client_embeddings, client_ids)

INFO:baseline.aggregated_features_baseline.create_embeddings:Saving embeddings


In [20]:
embeddings_dir = Path("embeddings_svd")
client_ids_svd = np.load(embeddings_dir / "client_ids.npy")
embeddings_svd = np.load(embeddings_dir / "embeddings.npy")

embedding_svd_df = pd.DataFrame(embeddings_svd, index=client_ids_svd)
embedding_svd_df.reset_index(inplace=True)
embedding_svd_df.rename(columns={'index': 'client_id'}, inplace=True)
embedding_svd_df.head()

/home/itsv.org.sv-services.at/tibor.cus@itsv.at/Projects/personal/recs2025/.venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/home/itsv.org.sv-services.at/tibor.cus@itsv.at/Projects/personal/recs2025/.venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,client_id,0,1,2,3,4,5,6,7,8,...,222,223,224,225,226,227,228,229,230,231
0,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,61,170.0,152.0,3.0,222.0,170.0,121.0,121.0,84.0,121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
embeddings_dir = Path("embeddings_handmade")
client_ids_hm = np.load(embeddings_dir / "client_ids.npy")
embeddings_hm = np.load(embeddings_dir / "embeddings.npy")

embedding_hm_df = pd.DataFrame(embeddings_hm, index=client_ids_hm)
embedding_hm_df.reset_index(inplace=True)
embedding_hm_df.rename(columns={'index': 'client_id'}, inplace=True)
embedding_hm_df.head()

/home/itsv.org.sv-services.at/tibor.cus@itsv.at/Projects/personal/recs2025/.venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/home/itsv.org.sv-services.at/tibor.cus@itsv.at/Projects/personal/recs2025/.venv/lib/python3.13/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,client_id,0,1,2,3,4,5,6,7,8,...,222,223,224,225,226,227,228,229,230,231
0,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,61,170.0,152.0,3.0,222.0,170.0,121.0,121.0,84.0,121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
